In [17]:
import math
import random
import numpy as np
from numpy import polyfit, poly1d
import argparse
from pathlib import Path
import shutil 
import os 
import sys
import re
import yaml
from gen_structures import mol_freq
from matplotlib import pyplot as plt
from matplotlib.pyplot import MultipleLocator
from prettytable import PrettyTable

import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.pyplot import MultipleLocator
from matplotlib import cycler
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

from sympy.physics.quantum.cg import CG
from sympy import S


from ase import Atoms
from ase.io import read
from ase.visualize import view
from ase.build import fcc111, add_adsorbate
from ase.io import write
from ase.io import vasp

class Eigenstates(object):
    """
        the object corresponds to analyze the Eigenstates.
    """ 
    def __init__(self, filename):
        self.filename = filename

    def read_energies_Pyscf(self):
        """
        read Energies from the file filename, get the eigenvalues list of E, and the number of eigenstates n )  
        """
        E = []   #B_kq,where k = 2
        with open(self.filename,"r", encoding="utf-8") as file:
            while True:
                line = file.readline()
                if not line:
                    break
                if len(line.split()) == 6 and " ".join(line.split())== "Nj eigevanlues of the electronic H": #find the correct position
                    # find the correct position,
                    for i in range(0,5):
                        line = file.readline()
                    i = 0
                    while True:
                        line = file.readline()
                        
                        if line.split()[0] == "|":
                            #print(line.split())
                            E += [float(line.split()[3])]
                            i += 1 
                        else:
                            self.Vaules = E
                            self.n_states = i
                            J = float((self.n_states-1)/2)
                            self.J = J
                            return 1
            print("not find eigenvaules")
            return 0

    def read_wavefunction_Pyscf(self):
        """
        read wavefunction as expansion coefficient of |JM>  from the file filename, 
        return the expansion coefficient matrix A,B(\Psi = (A*+Bi)*\JM) )  
        """
        pattern = re.compile(r"[+-]?\d+\.\d+|[+-]?\d+")   #determine the re pattern of number needed to be found 
        A = np.empty([self.n_states,self.n_states], dtype = float)
        B = np.empty([self.n_states,self.n_states], dtype = float)
        with open(self.filename,"r", encoding="utf-8") as file:
            while True:
                line = file.readline()
                if not line:
                    break
                if len(line.split()) == 7 and " ".join(line.split()) == "H eigenstates in the Jz eigenstates basis":
                    #for i in range(0,1):
                    line = file.readline()
                    for k in range(0,self.n_states):
                        for i in range(0,5):
                            line = file.readline()
                        for i in range(0,16):
                            line = file.readline()
                            A[k,i] = float(line.split('|')[2])
                            B[k,i] = float(line.split('|')[3])
                    C = A + B*1j  
                    np.savetxt("wavefunction_C",C)  
                    self.A = A
                    self.B = B  

filename="inpfile/eqout"
Eigen = Eigenstates(filename)
Eigen.read_energies_Pyscf()
#print(Eigen.Vaules,Eigen.n_states)
Eigen.read_wavefunction_Pyscf()
#print(Eigen.A,Eigen.B)

<>:74: SyntaxWarning: invalid escape sequence '\P'
<>:74: SyntaxWarning: invalid escape sequence '\P'
C:\Users\10122\AppData\Local\Temp\ipykernel_35960\1455358887.py:74: SyntaxWarning: invalid escape sequence '\P'
  """


In [ ]:
import pymol
from pymol import cmd
import random

# 1. 启动PyMOL
# 这个函数会启动PyMOL的GUI窗口
pymol.finish_launching()

cmd.load("inpfile/eq_opt.xyz", "my_molecule")

# 3. 获取分子中的原子总数
num_atoms = cmd.count_atoms("my_mol")
print(f"原子总数: {num_atoms}")

# 4. 为每个原子生成一个数值（这里演示用随机数，实际可替换为你的数据）
#    例如：从外部文件读取的列表，或通过计算得到的值
values = [random.uniform(0, 100) for _ in range(num_atoms)]

# 5. 使用 alter 命令将数值赋值给原子的 b 因子
#    注意：alter 会遍历每个原子，使用临时变量 'b' 存储数值
#    values 列表需要与原子顺序严格对应
cmd.alter("my_mol", f"b = {values.pop(0)}")  # 逐原子赋值

# 或者更安全的方式：一次性传入所有值（避免 pop 顺序问题）
# 但 alter 不支持直接传列表，所以推荐用循环或重新构建选择
# 下面的方法更可靠：通过索引遍历
# 先清空 b 值
cmd.alter("my_mol", "b = 0.0")
for i, val in enumerate(values):
    # 选择第 i 个原子（索引从0开始）
    sel = f"my_mol and index {i+1}"
    cmd.alter(sel, f"b = {val}")

# 6. 重建模型，使修改生效
cmd.rebuild()

# 7. 设置显示样式（例如球棍模型）
cmd.hide("everything", "my_mol")
cmd.show("sticks", "my_mol")
cmd.show("spheres", "my_mol")
cmd.set("sphere_scale", 0.3, "my_mol")

# 8. 基于 b 因子进行梯度着色
#    参数：spectrum <属性>, <选择>, <颜色图谱>, [范围]
#    b 因子范围：默认自动计算最小值/最大值
cmd.spectrum("b", "rainbow", "my_mol", minimum=0, maximum=100)
# 常用图谱: 'rainbow', 'red_white_blue', 'yellow_red', 'blue_white_red' 等

# 9. 调整视角
cmd.bg_color("white")
cmd.orient()
cmd.zoom(buffer=2)

# 保持窗口打开
pymol.finish_launching()

Qt not available (pymol.Qt), using GLUT/Tk interface


: 